# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a modern FAIR dataset—including multi-table record sets, data fields, and columns—using the [mlcroissant](https://github.com/mlcommons/croissant) library. All entities (record sets, fields, columns) are referenced by their Croissant schema `@id` for reproducibility and future automation.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant metadata provides information about the record sets and data structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset and inspect basic metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Let's review the available record sets (`cr:RecordSet`), the fields (columns) within each record set, and their `@id`s. This is crucial for referencing and manipulating the right parts of the dataset.

In [ ]:
# List all record sets and their fields by @id

record_sets = metadata.recordSets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields / Columns:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, dataType: {f.dataType})")
    print("\n")

For convenience, let's print a sample row for each record set using their `@id`:

In [ ]:
# List the first record of each record set (referencing by @id)
for rs in record_sets:
    print(f"First record of {rs.name} (@id: {rs.id}):")
    records_iter = dataset.records(record_set=rs.id)
    try:
        first_row = next(records_iter)
        print(first_row)
    except StopIteration:
        print("  No records available.")
    print("-")

## 3. Data Extraction
Let's extract the data from a main record set. In this dataset, there is usually a primary table containing the mass spectrometry results. We'll refer to all record sets by their `@id`, as shown in the previous overview. 

For the examples below, we select the first record set. If there are multiple record sets, you can adapt the logic to choose others as needed.

In [ ]:
# We'll extract all record sets to separate DataFrames
dfs = {}  # Map of record set @id to DataFrame

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dfs[rs.id] = df
    print(f"Loaded: {rs.name} (@id: {rs.id}) with {len(df)} rows and columns: {df.columns.tolist()}")

For demonstration, let's peek at the head (first rows) of the main data table for further analysis.

In [ ]:
# Choose the main data record set id (adjust if multiple main tables exist)
main_rs = record_sets[0].id
main_df = dfs[main_rs]
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate some basic processing. We'll:
- Select a numeric column by its `@id`
- Remove outliers and normalize values
- Optionally group by a categorical field

**Note:** You should adjust the field/column `@id` below based on the columns shown above for your actual analysis.

In [ ]:
# Example: Filter and normalize protein "MW_kDa" (molecular weight in kiloDaltons) if present

# List available columns for reference
print("Main DataFrame columns:", main_df.columns.tolist())

# Select a numeric field (@id or column name, adapt if your field differs)
# Below is a common pattern for protein datasets; replace as appropriate
# Suppose 'MW_kDa' or similar is present
candidate_numeric_columns = [c for c in main_df.columns if 'mw' in c.lower() or 'weight' in c.lower() or 'kda' in c.lower()]

if candidate_numeric_columns:
    numeric_field = candidate_numeric_columns[0]
else:
    numeric_field = main_df.select_dtypes('number').columns[0]  # fallback

print(f"Using numeric field: {numeric_field}")

threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a group field if any categorical fields are present
categorical_fields = main_df.select_dtypes('object').columns.tolist()
group_field = None
if len(categorical_fields) > 0:
    group_field = categorical_fields[0]

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and the grouped mean if grouping was possible. This helps assess overall trends and quality of the data.

In [ ]:
import matplotlib.pyplot as plt

# Distribution plot for the normalized numeric field
plt.figure(figsize=(7,4))
filtered_df[f"{numeric_field}_normalized"].hist(bins=30)
plt.title(f'Normalized Distribution of {numeric_field}')
plt.xlabel(f'{numeric_field} (normalized)')
plt.ylabel('Count')
plt.show()

# Grouped bar plot if grouped_df exists
if 'grouped_df' in locals():
    grouped_df.plot(kind='bar', x=group_field, y=numeric_field, legend=False)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'{numeric_field} (mean)')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored a highly structured FAIR dataset using the `mlcroissant` library, referencing all entities by their Croissant schema `@id`. We demonstrated how to:
- List record sets and their fields
- Load records into DataFrames
- Filter and normalize numeric data
- Visualize the results

You can further extend this notebook to perform domain-specific analyses, handle specific normalization, or cross-link with annotation metadata as needed. For reproducible science, always reference Croissant fields by `@id` as shown.